# Telemetry Anomaly Monitor

End-to-end reproduction of the deployed project. This notebook loads the supplied OPS-SAT segment dataset, explores it, trains ExtraTrees and Isolation Forest models, selects an operating threshold using validation data, evaluates the official holdout, and saves deployment artifacts.

In [ ]:
import json, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from pathlib import Path

ROOT=Path.cwd()
if not (ROOT/'data/dataset.csv').exists(): ROOT=Path('..')
df=pd.read_csv(ROOT/'data/dataset.csv')
df.head()

## 1. Dataset profile

In [ ]:
print('Shape:', df.shape)
print('Anomaly counts:\n', df.anomaly.value_counts())
print('Channels:', df.channel.nunique())
print('Official train/test sizes:', (df.train==1).sum(), (df.train==0).sum())
df.describe(include='all').T

In [ ]:
df.groupby('channel').anomaly.agg(['count','sum','mean']).sort_values('mean',ascending=False)

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
ax.scatter(df['mean'],df['std'],c=df['anomaly'],alpha=.5,s=15)
ax.set_xlabel('Mean'); ax.set_ylabel('Std'); ax.set_title('Mean vs standard deviation')
plt.show()

## 2. Feature preparation

The identifier and labels are excluded. The supplied `train` column is not treated as a model feature. Channel is one-hot encoded and numeric values are median-imputed.

In [ ]:
TARGET='anomaly'
DROP=['segment','anomaly','train']
features=[c for c in df.columns if c not in DROP]
cat=['channel']
num=[c for c in features if c!='channel']
train=df[df.train==1].copy(); test=df[df.train==0].copy()
tr,val=train_test_split(train,test_size=.20,stratify=train[TARGET],random_state=42)
pre=ColumnTransformer([('num',SimpleImputer(strategy='median'),num),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
clf=ExtraTreesClassifier(n_estimators=500,min_samples_leaf=2,class_weight='balanced',random_state=42,n_jobs=-1)
pipe=Pipeline([('preprocessor',pre),('classifier',clf)])
pipe.fit(tr[features],tr[TARGET])

## 3. Threshold selection

In [ ]:
val_prob=pipe.predict_proba(val[features])[:,1]
thresholds=np.linspace(.15,.85,141)
f1s=[f1_score(val[TARGET],val_prob>=t) for t in thresholds]
best=float(thresholds[int(np.argmax(f1s))])
print('Selected threshold:',best,'validation F1:',max(f1s))

## 4. Final model and official holdout evaluation

In [ ]:
pipe.fit(train[features],train[TARGET])
test_prob=pipe.predict_proba(test[features])[:,1]
pred=(test_prob>=best).astype(int)
metrics={
'accuracy':accuracy_score(test[TARGET],pred),
'precision':precision_score(test[TARGET],pred),
'recall':recall_score(test[TARGET],pred),
'f1':f1_score(test[TARGET],pred),
'roc_auc':roc_auc_score(test[TARGET],test_prob),
'pr_auc':average_precision_score(test[TARGET],test_prob)}
print(pd.Series(metrics))
print('Confusion matrix:\n',confusion_matrix(test[TARGET],pred))

In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
ax.hist(test_prob[test[TARGET].values==0],bins=30,alpha=.6,label='Nominal')
ax.hist(test_prob[test[TARGET].values==1],bins=30,alpha=.6,label='Anomaly')
ax.axvline(best,linestyle='--',label=f'Threshold={best:.3f}')
ax.set_xlabel('Predicted anomaly probability'); ax.set_ylabel('Count'); ax.legend(); ax.set_title('Holdout risk distribution'); plt.show()

## 5. Optional Isolation Forest

In [ ]:
iso_pre=ColumnTransformer([('num',SimpleImputer(strategy='median'),num),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
iso=Pipeline([('preprocessor',iso_pre),('detector',IsolationForest(n_estimators=300,contamination='auto',random_state=42,n_jobs=-1))])
iso.fit(train[features])
print('Isolation Forest trained. It is intended as a complementary unsupervised detector, not as the headline labeled metric.')

## 6. Save artifacts used by Streamlit and Flask

In [ ]:
(ROOT/'model').mkdir(exist_ok=True)
joblib.dump(pipe,ROOT/'model/telemetry_extratrees.joblib')
joblib.dump(iso,ROOT/'model/telemetry_isolation_forest.joblib')
metadata={**{k:float(v) for k,v in metrics.items()},'threshold':best,'features':features}
with open(ROOT/'model/metadata.json','w') as f: json.dump(metadata,f,indent=2)
metadata